## Importing Libraries

In [74]:
import pandas  as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
import biogeme.expressions as exp
import biogeme.results as res

import seaborn as sns
import matplotlib.pyplot as plt

import re
import os
import shutil
import copy
import warnings
import functools
import contextlib
import time

from decimal import Decimal
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

from sklearn.metrics import confusion_matrix, classification_report, fbeta_score


In [2]:
plt.style.use('dark_background')
pd.set_option("display.precision", 2)
seed = 1
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

## Data Loading

In [93]:
# Data Loading
df = pd.read_csv('data/data_incentive_attitudes_residencedata.csv')

# Some basic data preprocessing steps
df = df.drop(['X', 'trip','value','long_o', 'lat_o', 'long_d.x', 
              'lat_d.x', 'pincode', 'address','Prefecture', 'City', 
              'Town','lat_d.return','long_d.return', 'dest_inside'], axis='columns')

df = df.rename(columns={'pubcost':'traincost', 
                   'pubtime':'traintime', 
                   'pubavail':'trainavail',
                    'bicycleincentive':'bikeincentive'})

# Replacing 'pub' with 'train' in `mode` column.
selected_mode = df['mode'].to_numpy()
df['mode'] = np.where(selected_mode=='pub', 'train', selected_mode)

purpose_code_dict = {100: 'Commuting to work / school',
                    101: 'Go Home',
                    200: 'Shopping for daily necessities',
                    201: 'Shopping other than daily necessities',
                    202: 'Meals and entertainment',
                    300: 'business',
                    400: 'Outpatient',
                    500: 'Pick-up and drop-off',
                    600: 'Sightseeing / Leisure',
                    998: 'others'}

attitudinal_variables = [x for x in df.columns if bool(re.search(r'\d', x))]
ohe_generic_variables = dict()

## Preprocessing
- One hot encoding of categorical features
- Removing unwanted columns

In [94]:
# utility functions for extracting time-zones and trip duration information from arrival and departure times of a trip.

def get_trip_duration(df):
    """
    Returns the total minutes a trip lasted given its departure and arrival datetimes.
    """
    dep_datetime = datetime.strptime(df['departure_time'], '%m/%d/%Y %H:%M')
    arr_datetime = datetime.strptime(df['arrival_time'], '%m/%d/%Y %H:%M')
    
    return int((arr_datetime - dep_datetime).total_seconds()/60)

def get_day_zones(trip_datetime):
    """
    Divides the time into 4 zones and ordinally encoding them.
    Early morning (00:00 to 6:00) - 1
    AM peak (6:00 to 10:00) - 2
    Off peak (10:00 to 16:00) - 3
    PM peak (16:00 to 20:00) - 4
    Evening (20:00 to 00:00) - 5
    """
    dep_hrs, dep_mins = [int(val) for val in trip_datetime.split(' ')[1].split(':')]
    if dep_hrs >= 0 and dep_hrs < 6:
        return 1
    elif dep_hrs >= 6 and dep_hrs < 10:
        return 2
    elif dep_hrs >= 10 and dep_hrs < 16:
        return 3
    elif dep_hrs >= 16 and dep_hrs < 20:
        return 4
    else:
        return 5
    
# Modifying incentive columns, limiting incentives to incentive zones
df['walkavail'] = np.where(df['mode']=='walk', 1, np.where(df['cardistance']<=7, 1, 0))
df['busincentive'] = np.where(df['incentivezone']==1, df['busincentive'], 0)
df['carincentive'] = np.where(df['incentivezone']==1, df['carincentive'], 0)
df['trainincentive'] = np.where(df['incentivezone']==1, df['trainincentive'], 0)
df['bikeincentive'] = np.where(df['incentivezone']==1, df['bikeincentive'], 0)
df['walkincentive'] = np.where(df['incentivezone']==1, df['walkincentive'], 0)
df['motorincentive'] = np.where(df['incentivezone']==1, df['motorincentive'], 0)

ohe = OneHotEncoder()
df['high_income'] = np.where(df['INCOME'] >= 5, 1, 0)

# `job_type` column
job_type_sparse_matrix = ohe.fit_transform(df['job_type'].to_numpy().reshape(-1, 1)).toarray()
job_type_column_names = ['job_type_'+job for job in df['job_type'].unique()]
job_type_df = pd.DataFrame(data=job_type_sparse_matrix, columns=job_type_column_names)
job_type_df = job_type_df.drop(job_type_df.columns[0], axis=1)
ohe_generic_variables['jobs'] = list(job_type_df.columns)

# `information` column
info_sparse_matrix = ohe.fit_transform(df['information'].to_numpy().reshape(-1, 1)).toarray()
info_df = pd.DataFrame(data=info_sparse_matrix, columns=ohe.categories_[0])
info_df = info_df.drop(info_df.columns[0], axis=1)
ohe_generic_variables['info'] = list(info_df.columns)

# `SEX` column
SEX_sparse_matrix = ohe.fit_transform(df['SEX'].to_numpy().reshape(-1, 1)).toarray()
SEX_df = pd.DataFrame(data=SEX_sparse_matrix, columns=['Male', 'Female'])
SEX_df = SEX_df.drop(SEX_df.columns[0], axis=1)

# `Purpose` column
Purpose_sparse_matrix = ohe.fit_transform(df['Purpose'].to_numpy().reshape(-1, 1)).toarray()
Purpose_column_names = ['Purpose_'+str(purpose_code) for purpose_code in df['Purpose'].unique()]
Purpose_df = pd.DataFrame(data=Purpose_sparse_matrix, columns=Purpose_column_names)
Purpose_df = Purpose_df.drop('Purpose_999', axis=1)
ohe_generic_variables['purposes'] = list(Purpose_df.columns)

# Getting the trip duration in minutes
df['trip_duration'] = df[['departure_time', 'arrival_time']].T.apply(get_trip_duration)

# Converting departure time in time zones and one-hot encoding it
dep_time_zones_sparse_matrix = ohe.fit_transform(df['departure_time'].apply(get_day_zones).to_numpy().reshape(-1, 1)).toarray()
dep_time_zones_names = ['Early_morning_departure', 'AM_peak_departure', 'Off_peak_departure', 'PM_peak_departure', 'Night_departure']
dep_time_zones_df = pd.DataFrame(data=dep_time_zones_sparse_matrix, columns=dep_time_zones_names)
dep_time_zones_df = dep_time_zones_df.drop('Night_departure', axis=1)
ohe_generic_variables['departures'] = list(dep_time_zones_df.columns)

# combining all the one-hot encoded dataframes with the main dataframe
df2 = pd.concat([df, dep_time_zones_df, Purpose_df, SEX_df, job_type_df, info_df], axis=1).drop(
    ['departure_time', 'arrival_time', 'Purpose', 'SEX', 'job_type'], axis='columns')

# Removing some unnecessary columns
df2 = df2.drop(['user_id', 'trip_id', 'recco', 'information', 'incentivezone', 'task', 'income_con'], axis='columns')

df2.head()

,buscost,bustime,carcost,cartime,cardistance,traincost,traintime,walktime,walkcost,biketime,...,job_type_Part time job,job_type_Management executive,job_type_civil servant,job_type_Housewife,job_type_Self employed/ Freelance,job_type_Others,job_type_Unemployed,no info,only enviro,only health
0,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,690,66.2,128.6,38.96,12.86,240,44.9,154.32,0,51.44,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,500,56.2,118.5,29.53,11.85,400,59.0,142.20,0,47.40,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


## Splitting the data

In [95]:
def get_CBD_data_split(df, drop_att_vars=True, apply_smote=False, apply_tomek_links=False,
                      apply_standardization=False, cols_to_standardize=[]):
    if drop_att_vars:
        df = df.drop(attitudinal_variables, axis='columns')
        
        oCBD_df = df.loc[df['address_inside'] == False]
        iCBD_df = df.loc[(df['address_inside'] == True)]
        
        label_mapping_dict = {'bus': 1, 'car': 2, 'train': 3, 'walk': 4, 'bike': 5, 'motor': 6}
                
        y_oCBD = np.array([label_mapping_dict[label] for label in oCBD_df['mode']])
        X_oCBD = oCBD_df.drop('mode', axis='columns')
        y_iCBD = np.array([label_mapping_dict[label] for label in iCBD_df['mode']])
        X_iCBD = iCBD_df.drop('mode', axis='columns')
        
        if apply_smote:
            smote = SMOTE(random_state=500)
            X_oCBD, y_oCBD = smote.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = smote.fit_resample(X_iCBD, y_iCBD)
        
        if apply_tomek_links:
            tomek = TomekLinks()
            X_oCBD, y_oCBD = tomek.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = tomek.fit_resample(X_iCBD, y_iCBD)
        
        if apply_standardization:
            ss =  StandardScaler()
            X_oCBD_scaled_arr = ss.fit_transform(X_oCBD[cols_to_standardize])
            X_iCBD_scaled_arr = ss.transform(X_iCBD[cols_to_standardize])
            X_oCBD_scaled = pd.DataFrame(X_oCBD_scaled_arr, columns=cols_to_standardize)
            X_iCBD_scaled = pd.DataFrame(X_iCBD_scaled_arr, columns=cols_to_standardize)
            X_oCBD_scaled.set_index(X_oCBD.index, inplace=True)
            X_iCBD_scaled.set_index(X_iCBD.index, inplace=True)
            X_oCBD.iloc[0:X_oCBD.shape[0], X_oCBD.columns.get_indexer(cols_to_standardize)] = X_oCBD_scaled
            X_iCBD.iloc[0:X_iCBD.shape[0], X_iCBD.columns.get_indexer(cols_to_standardize)] = X_iCBD_scaled
        
        return ((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict
    

In [96]:
output_categories = ['bus', 'car', 'train', 'walk', 'bike', 'motor']

asc_variable_names = {
    'bus': ['buscost', 'bustime'],
    'car': ['carcost', 'cartime'],
    'train': ['traincost', 'traintime'], 
    'walk': ['walkcost', 'walktime'],
    'bike': [ 'bikecost', 'biketime'],
    'motor': ['motorcost', 'motortime']
}

avail_variable_names = {
    'bus': 'busavail',
    'train': 'trainavail',
    'walk': 'walkavail',
    'bike': 'bikeavail',
    'motor': 'motoravail'
}

In [97]:
((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict = get_CBD_data_split(df2, drop_att_vars=True)

In [98]:
X_oCBD['mode'] = y_oCBD
X_oCBD['caravail'] = np.ones(X_oCBD.shape[0])
X_iCBD['mode'] = y_iCBD
X_iCBD['caravail'] = np.ones(X_iCBD.shape[0])

In [99]:
bio_db = db.Database("Outside CBD", X_oCBD.drop('address_inside', axis='columns'))
bio_db_test = db.Database("Inside CBD", X_iCBD.drop('address_inside', axis='columns'))

## Modelling

In [100]:
globals().update(bio_db.variables)

In [101]:
ind_spec_vars = []

In [102]:
ASC_BUS   = exp.Beta('ASC_BUS',0,None ,None ,0)
ASC_TRAIN = exp.Beta('ASC_TRAIN',0,None ,None ,0)
ASC_WALK  = exp.Beta('ASC_WALK',0,None ,None ,0)
ASC_BIKE = exp.Beta('ASC_BIKE', 0, None, None, 0)
ASC_MOTOR = exp.Beta('ASC_MOTOR', 0, None, None, 0)

B_TIME    = exp.Beta('B_TIME',0,None ,None ,0)
B_COST    = exp.Beta('B_COST',0,None ,None ,0)
#B_INC    = exp.Beta('B_INC',0,None ,None ,0)

B_IS = {}
for ind_spec_var in ind_spec_vars:
    if ind_spec_var == 'nearest_bus_dist':
        B_IS[f'B_{ind_spec_var}_bus'] = exp.Beta(f'B_{ind_spec_var}_bus', 0, None, None, 0)
    elif ind_spec_var == 'nearest_train_dist':
        B_IS[f'B_{ind_spec_var}_train'] = exp.Beta(f'B_{ind_spec_var}_train', 0, None, None, 0)
    else:
        for alternative in output_categories:
            if alternative == 'car':
                continue
            B_IS[f'B_{ind_spec_var}_{alternative}'] = exp.Beta(f'B_{ind_spec_var}_{alternative}', 0, None, None, 0)

In [103]:
U_CAR = B_TIME*cartime + B_COST*carcost
U_BUS = B_TIME*bustime + B_COST*buscost + ASC_BUS
U_TRAIN = B_TIME*traintime + B_COST*traincost + ASC_TRAIN
U_WALK = B_TIME*walktime + B_COST*walkcost + ASC_WALK
U_BIKE = B_TIME*biketime + B_COST*bikecost + ASC_BIKE
U_MOTOR = B_TIME*motortime + B_COST*motorcost + ASC_MOTOR

for is_var in ind_spec_vars:
    if is_var == 'nearest_bus_dist':
        U_BUS += nearest_bus_dist*B_IS[f'B_{is_var}_bus']
    elif is_var == 'nearest_train_dist':
        U_TRAIN += nearest_train_dist*B_IS[f'B_{is_var}_train']
    else:
        U_BUS += bio_db.variables[is_var] * B_IS[f'B_{is_var}_bus']
        U_TRAIN += bio_db.variables[is_var] * B_IS[f'B_{is_var}_train']
        U_WALK += bio_db.variables[is_var] * B_IS[f'B_{is_var}_walk']
        U_BIKE += bio_db.variables[is_var] * B_IS[f'B_{is_var}_bike']
        U_MOTOR += bio_db.variables[is_var] * B_IS[f'B_{is_var}_motor']

In [104]:
utilities = {
    1 : U_BUS,
    2 : U_CAR,
    3 : U_TRAIN,
    4 : U_WALK,
    5 : U_BIKE,
    6 : U_MOTOR
}

avails = {
    1 : busavail,
    2 : caravail,
    3 : trainavail,
    4 : walkavail,
    5 : bikeavail,
    6 : motoravail
}

In [105]:
avail_choice_ratio_dict_train = {output_categories[int(key)-1]: np.round(value[1]/value[0], 2) for key, value in bio_db.choiceAvailabilityStatistics(avails, mode).items()}
print(f"Training data's mode availability vs choice ratio: \n{avail_choice_ratio_dict_train}\n")

avail_choice_ratio_dict_test = {output_categories[int(key)-1]: np.round(value[1]/value[0], 2) for key, value in bio_db_test.choiceAvailabilityStatistics(avails, mode).items()}
print(f"Testing data's mode availability vs choice ratio: \n{avail_choice_ratio_dict_test}")

Training data's mode availability vs choice ratio: 
{'bus': 12.48, 'car': 1.43, 'train': 7.69, 'walk': 5.05, 'bike': 3.99, 'motor': 4.75}

Testing data's mode availability vs choice ratio: 
{'bus': 14.57, 'car': 1.79, 'train': 11.58, 'walk': 7.17, 'bike': 3.07, 'motor': 1.99}


In [106]:
def move_file(file_path, dir_path, new_name):
    new_path = os.path.join(dir_path, new_name)
    if os.path.exists(new_path):
        os.remove(new_path)
    shutil.move(file_path, new_path)

def reorganize_res_files(dir_path):
    
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
    
    if os.path.exists("biogeme.toml"):
        biogeme_param_file = os.path.basename("biogeme.toml")
        move_file("biogeme.toml", dir_path, "biogeme.toml")
    if os.path.exists("__.iter"):
        model_param_file = os.path.basename("__.iter")
        move_file("__.iter", dir_path, "model_params.iter")
    if os.path.exists("~00.html"):
        html_file = os.path.basename("~00.html")
        move_file("~00.html", dir_path, "model_report.html")
    if os.path.exists("~00.pickle"):
        model_pickle_file = os.path.basename("~00.pickle")
        move_file("~00.pickle", dir_path, "model.pickle")
    

In [107]:
logprob = models.loglogit(utilities, avails, mode)
model_note = "Using only travel time and cost variables"
biogeme  = bio.BIOGEME(bio_db, logprob, user_notes=model_note)

biogeme.modelName = ""
biogeme.algorithm_name = 'LS-BFGS'
biogeme.tolerance = 1e-8
biogeme.maxiter = 1000
biogeme.generate_html = True
biogeme.only_robust_stats = False
results = biogeme.estimate()

reorganize_res_files("res/classical/cost_time_only_model")

File biogeme.toml has been created
